# TimelyMT Research MVP: Kaggle GPU Runner
Run cells in order on a Kaggle GPU notebook with Internet enabled. This notebook only orchestrates the existing `timelymt.research.cli`; it contains no research logic. Upload the current repository tree as a Kaggle Dataset to preserve local atomic pseudo-label artifacts, or set `REPO_URL` to clone a clean repository.

## ENVIRONMENT SETUP

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = ""  # Optional: clean clone. Prefer an uploaded repo Dataset to retain partial artifacts.
WORK_REPO = Path("/kaggle/working/TimelyMT")
INFERENCE_BATCH_SIZE = 3  # Reduce to 2 or 1 if CUDA runs out of memory.

if not (WORK_REPO / "pyproject.toml").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(WORK_REPO)], check=True)
    else:
        candidates = sorted(Path("/kaggle/input").rglob("pyproject.toml"))
        sources = [path.parent for path in candidates if (path.parent / "src/timelymt/research/cli.py").exists()]
        if len(sources) != 1:
            raise RuntimeError(f"Expected one uploaded TimelyMT repository, found: {sources}")
        shutil.copytree(sources[0], WORK_REPO, dirs_exist_ok=True)

os.chdir(WORK_REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "transformers>=4.57.6,<5.0.0", "sentencepiece", "sacrebleu",
                "scikit-learn", "joblib", "huggingface_hub"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

def cli(*args):
    command = [sys.executable, "-u", "-m", "timelymt.research.cli", *args]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=WORK_REPO, check=True)

print(f"Repository: {WORK_REPO}")
print(f"Inference batch size: {INFERENCE_BATCH_SIZE}")

## MODEL/CACHE SETUP

In [ ]:
MODEL_ID = "VietAI/envit5-translation"
MODEL_REVISION = "840bc88104d5a4277af740eaedb024df8c3093e7"
HF_HOME = Path("/kaggle/temp/huggingface")  # Runtime-only; translator weights are not exported.
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.update({
    "HF_HOME": str(HF_HOME),
    "HF_HUB_CACHE": str(HF_HOME / "hub"),
    "TOKENIZERS_PARALLELISM": "false",
})
from huggingface_hub import snapshot_download
snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, cache_dir=os.environ["HF_HUB_CACHE"])
(WORK_REPO / "outputs/translator/cache").mkdir(parents=True, exist_ok=True)
print(f"Pinned model cached under {HF_HOME}")
print(f"Persistent translation cache: {WORK_REPO / 'outputs/translator/cache'}")

## PRECHECK

In [ ]:
import json, torch, transformers
from timelymt.research.cli import _manifests
from timelymt.translator.envit5 import load_config, resolve_device

config = load_config(WORK_REPO / "configs/translator/envit5.json")
_manifests()
assert config.model_id == MODEL_ID and config.model_revision == MODEL_REVISION and config.frozen
assert tuple(map(int, transformers.__version__.split(".")[:2])) >= (4, 57)
assert int(transformers.__version__.split(".")[0]) < 5
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before continuing")
print({"cuda": torch.cuda.get_device_name(0), "device_policy": config.device_policy,
       "resolved_device": resolve_device(config.device_policy), "transformers": transformers.__version__,
       "model_id": config.model_id, "model_revision": config.model_revision, "frozen": config.frozen})
cli("--help")

## TIMELYMT TRAIN PSEUDO

In [ ]:
cli("pseudo", "--split", "train", "--batch-size", str(INFERENCE_BATCH_SIZE))

## TIMELYMT DEV PSEUDO

In [ ]:
cli("pseudo", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE))

## MU TRAIN/DEV SUPERVISION

In [ ]:
cli("mu-supervision", "--split", "train", "--batch-size", str(INFERENCE_BATCH_SIZE))
cli("mu-supervision", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE))

## VALIDATE

In [ ]:
cli("validate-pseudo", "--split", "train")
cli("validate-pseudo", "--split", "dev")
cli("validate-mu", "--split", "train")
cli("validate-mu", "--split", "dev")

## TRAIN P0/P1/P2

In [ ]:
for variant in ("P0", "P1", "P2"):
    cli("train", "--pseudo-labels", "data/policy/pseudo_labels/train/manifest.json", "--variant", variant)

## TRAIN MU

In [ ]:
cli("train-mu", "--pseudo-labels", "data/policy/mu_zhang2020/train/manifest.json")

## DEV BASELINES

In [ ]:
FIXED = ["fixed_n_4", "fixed_n_8", "fixed_n_12", "fixed_time_1600",
         "fixed_time_3200", "fixed_time_4800"]
STYLE = ["local_agreement_style_k2", "local_agreement_style_k3"]
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", *FIXED)
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", *STYLE)

## DEV LA-2

In [ ]:
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", "local_agreement_la2")

## DEV MU ROLLOUT

In [ ]:
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", "mu_zhang2020")

## DEV LEARNED ROLLOUT

In [ ]:
LEARNED = [f"learned_{variant}_{threshold:.2f}" for variant in ("P0", "P1", "P2")
           for threshold in (0.30, 0.40, 0.50, 0.60, 0.70)]
cli("rollout", "--split", "dev", "--batch-size", str(INFERENCE_BATCH_SIZE), "--strategies", *LEARNED)

## DEV EVALUATE

In [ ]:
BASELINES = FIXED + STYLE + ["local_agreement_la2", "mu_zhang2020"]
cli("evaluate", "--split", "dev", "--strategies", *(BASELINES + LEARNED))

## DEV SELECT

In [ ]:
cli("select")

## FREEZE

In [ ]:
cli("freeze")

## EXPORT ARTIFACTS

In [ ]:
import tarfile
EXPORT = Path("/kaggle/working/timelymt-research-mvp-artifacts.tar.gz")
ARTIFACT_DIRS = [
    Path("data/policy/pseudo_labels"),
    Path("data/policy/mu_zhang2020"),
    Path("checkpoints/policy"),
    Path("outputs/experiments/research-mvp"),
]
with tarfile.open(EXPORT, "w:gz") as archive:
    for relative in ARTIFACT_DIRS:
        source = WORK_REPO / relative
        if not source.exists():
            raise RuntimeError(f"Missing required artifact directory: {source}")
        archive.add(source, arcname=relative.as_posix())
print(f"Download: {EXPORT}")
print("Translator model/cache weights under /kaggle/temp are intentionally not packaged.")

# STOP BEFORE TEST
The researcher's run ends at this cell. Do not add or execute any TEST rollout, TEST pseudo-label, TEST evaluation, selected-TEST, or TEST report command in this notebook. Download `/kaggle/working/timelymt-research-mvp-artifacts.tar.gz` (or the three artifact directories from `/kaggle/working/TimelyMT`) before ending the Kaggle session.